# 05 — Strategy Scanner

Searches across the S&P 500 for the best-performing ticker/parameter combination for a given strategy.

For each ticker, every parameter combination in a predefined grid is backtested. Results are ranked by the chosen metric.

Backtest period is fixed at 2015–2024 throughout.

In [1]:
import sys
sys.path.append('../src')

import threading
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import ipywidgets as widgets
from IPython.display import display, clear_output

from scanner import run_scan, fetch_sp500, PARAM_GRIDS, leveraged_metrics
from metrics import annualized_return, sharpe_ratio, max_drawdown, calmar_ratio

## Motivation

As the parameter sweeps in the previous notebooks suggest, the strategies presented in this project are not guaranteed to work at all. The interactive element below is a tool to gain a better understanding about which stock developments work best with each strategy.

## Load price data

All S&P 500 prices are downloaded once upfront. The scan then runs entirely on local data — no further network calls needed.

In [2]:
status_label = widgets.Label(value="Downloading S&P 500 price data...")
display(status_label)

prices_df = fetch_sp500()
status_label.value = f"Loaded {prices_df.shape[1]} tickers, {prices_df.shape[0]} trading days."

Label(value='Downloading S&P 500 price data...')

$Q: possibly delisted; no price data found  (1d 2015-01-01 -> 2024-01-01) (Yahoo error = "Data doesn't exist for startDate = 1420088400, endDate = 1704085200")
$SNDK: possibly delisted; no price data found  (1d 2015-01-01 -> 2024-01-01) (Yahoo error = "Data doesn't exist for startDate = 1420088400, endDate = 1704085200")
$GEV: possibly delisted; no price data found  (1d 2015-01-01 -> 2024-01-01) (Yahoo error = "Data doesn't exist for startDate = 1420088400, endDate = 1704085200")
$SOLV: possibly delisted; no price data found  (1d 2015-01-01 -> 2024-01-01) (Yahoo error = "Data doesn't exist for startDate = 1420088400, endDate = 1704085200")

4 Failed downloads:
['Q', 'SNDK', 'GEV', 'SOLV']: possibly delisted; no price data found  (1d 2015-01-01 -> 2024-01-01) (Yahoo error = "Data doesn't exist for startDate = 1420088400, endDate = 1704085200")


## Scanner UI

Select a strategy, a ranking method, and hit Run. The top 10 results appear as a table (start index 0) Click any row to plot that backtest.

In [ ]:
# ── widgets ──────────────────────────────────────────────────────────────────

strategy_dd = widgets.Dropdown(
    options=["MA Crossover", "Mean Reversion", "RSI"],
    value="MA Crossover",
    description="Strategy:",
    layout=widgets.Layout(width="260px"),
)

ranking_dd = widgets.Dropdown(
    options=["Total Return", "Sharpe Ratio", "Calmar Ratio", "Leveraged"],
    value="Sharpe Ratio",
    description="Rank by:",
    layout=widgets.Layout(width="260px"),
)

leverage_slider = widgets.FloatSlider(
    value=2.0, min=1.0, max=4.0, step=0.5,
    description="Leverage:",
    layout=widgets.Layout(width="340px"),
    disabled=True,
)

run_btn = widgets.Button(
    description="Run scan",
    button_style="primary",
    layout=widgets.Layout(width="120px"),
)

progress_bar = widgets.IntProgress(
    value=0, min=0, max=100,
    description="Progress:",
    layout=widgets.Layout(width="400px"),
    style={"bar_color": "#4a90d9"},
)

status_out   = widgets.HTML()
table_out    = widgets.HTML()
chart_out    = widgets.Output()

# enable/disable leverage slider based on ranking selection
def _on_ranking_change(change):
    leverage_slider.disabled = change["new"] != "Leveraged"

ranking_dd.observe(_on_ranking_change, names="value")

# ── state shared between threads ─────────────────────────────────────────────

_scan_results = {"df": None}

# ── plotting helper ───────────────────────────────────────────────────────────

def plot_result(row, strategy, ranking, leverage):
    """Four-panel chart for a single backtest result."""
    result = row["_result"]
    ticker = row["Ticker"]
    params = row["Params"]

    if ranking == "Leveraged":
        lev_eq, lev_ret = leveraged_metrics(result, leverage)
        eq  = lev_eq
        ret = lev_ret
        equity_label = f"Strategy {leverage}x leveraged"
    else:
        eq  = result["equity"]
        ret = result["strategy_ret"]
        equity_label = "Strategy"

    # drawdown series
    rolling_max = eq.cummax()
    drawdown    = (eq - rolling_max) / rolling_max

    fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True,
                              gridspec_kw={"height_ratios": [2, 1.2, 1]})

    # equity curve
    axes[0].plot(result.index, eq,                 label=equity_label, color="steelblue", linewidth=1.4)
    axes[0].plot(result.index, result["bh_equity"],label="Buy & Hold",  color="gray",      linewidth=1, alpha=0.7)
    axes[0].set_ylabel("Portfolio value ($)")
    params_str = ", ".join(f"{k}={v}" for k, v in params.items())
    axes[0].set_title(f"{strategy} — {ticker}   [{params_str}]")
    axes[0].legend()

    # price
    axes[1].plot(result.index, result["price"], color="black", linewidth=0.8)

    # mark buy/sell transitions
    pos = result["position"]
    buys  = result.index[pos.diff() > 0]
    sells = result.index[pos.diff() < 0]
    axes[1].scatter(buys,  result.loc[buys,  "price"], marker="^", color="seagreen", s=40, zorder=5, label="Buy")
    axes[1].scatter(sells, result.loc[sells, "price"], marker="v", color="tomato",   s=40, zorder=5, label="Sell")
    axes[1].set_ylabel("Price ($)")
    axes[1].legend(fontsize=8)

    # drawdown
    axes[2].fill_between(result.index, drawdown, 0, color="tomato", alpha=0.5)
    axes[2].set_ylabel("Drawdown")
    axes[2].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
    axes[2].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    fig.align_ylabels(axes)
    plt.tight_layout()
    plt.show()

    # metrics below chart
    print(f"  CAGR         : {annualized_return(eq):.2%}")
    print(f"  Sharpe       : {sharpe_ratio(ret):.2f}")
    print(f"  Max Drawdown : {max_drawdown(eq):.2%}")
    print(f"  B&H CAGR     : {annualized_return(result['bh_equity']):.2%}")
    print(f"  Beats B&H    : {annualized_return(eq) > annualized_return(result['bh_equity'])}")

# ── table interaction ─────────────────────────────────────────────────────────

row_selector = widgets.BoundedIntText(
    value=0, min=0, max=9,
    description="Plot row:",
    layout=widgets.Layout(width="180px"),
)

plot_btn = widgets.Button(
    description="Plot selected",
    button_style="",
    layout=widgets.Layout(width="140px"),
    disabled=True,
)

def _on_plot(b):
    df = _scan_results["df"]
    if df is None or df.empty:
        return
    idx = row_selector.value
    if idx >= len(df):
        return
    with chart_out:
        clear_output(wait=True)
        plot_result(
            df.iloc[idx],
            strategy_dd.value,
            ranking_dd.value,
            leverage_slider.value,
        )

plot_btn.on_click(_on_plot)

# ── scan execution ────────────────────────────────────────────────────────────

def _run_scan_thread():
    strategy = strategy_dd.value
    ranking  = ranking_dd.value
    leverage = leverage_slider.value

    n_tickers = prices_df.shape[1]
    n_params  = len(PARAM_GRIDS[strategy])
    total     = n_tickers * n_params
    progress_bar.max   = total
    progress_bar.value = 0

    status_out.value = f"Scanning {n_tickers} tickers × {n_params} parameter sets = {total} backtests..."
    table_out.value = "" # Clear previous table

    def _progress(value, maximum):
        progress_bar.value = value

    df = run_scan(
        strategy, ranking, prices_df,
        progress_cb=_progress,
        leverage=leverage,
    )

    _scan_results["df"] = df

    # Set up table
    display_cols = ["Ticker", "Params", "CAGR", "Sharpe", "Max Drawdown", "Calmar", "BH CAGR", "Beats B&H"]
    top10 = df.head(10)[display_cols].copy()
    top10["CAGR"]         = top10["CAGR"].map("{:.2%}".format)
    top10["Sharpe"]       = top10["Sharpe"].map("{:.2f}".format)
    top10["Max Drawdown"] = top10["Max Drawdown"].map("{:.2%}".format)
    top10["Calmar"]       = top10["Calmar"].map("{:.2f}".format)
    top10["BH CAGR"]      = top10["BH CAGR"].map("{:.2%}".format)

    table_out.value = top10.to_html(index=False)

    comb_x = df["Beats B&H"].sum()
    comb_y = len(df)
    
    best_per_stock = df.drop_duplicates(subset=["Ticker"])
    stock_x = best_per_stock["Beats B&H"].sum()
    stock_y = len(best_per_stock)

    status_out.value = (
        f"<br><b>Scan Complete.</b><br>"
        f"• <b>{comb_x} / {comb_y}</b> total parameter combinations beat Buy & Hold.<br>"
        f"• <b>{stock_x} / {stock_y}</b> unique stocks beat Buy & Hold (using their best parameters)."
    )

    run_btn.disabled  = False
    plot_btn.disabled = False
    row_selector.max  = min(9, len(df) - 1)

def _on_run(b):
    run_btn.disabled  = True
    plot_btn.disabled = True
    with chart_out:
        clear_output()
    thread = threading.Thread(target=_run_scan_thread, daemon=True)
    thread.start()

run_btn.on_click(_on_run)

# ── layout ────────────────────────────────────────────────────────────────────

controls = widgets.VBox([
    widgets.HBox([strategy_dd, ranking_dd]),
    widgets.HBox([leverage_slider]),
    widgets.HBox([run_btn, progress_bar]),
    status_out,
    table_out,
    widgets.HBox([row_selector, plot_btn]),
    chart_out,
])

display(controls)